In [14]:
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
import pandas as pd
df = pd.read_csv("D:/Learn_ML/ML_algos/datasets/BreastCancer.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df['diagnosis'] = df['diagnosis'].apply(lambda x : 1 if x == 'M' else -1)


In [5]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [8]:
df.isnull().sum()

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

In [9]:
N = len(df)
weights = np.ones(N) / N


In [13]:
print(sum(weights))

0.9999999999999938


In [17]:
y = df["diagnosis"]
X = df.drop('diagnosis', axis=1)


In [18]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , stratify=y)
print(X_train.shape , y_train.shape , X_test.shape , y_test.shape)

(455, 32) (455,) (114, 32) (114,)


In [19]:
def stump_predict(X, feature, threshold, left_label, right_label):

    preds = np.where(X[feature] <= threshold, left_label, right_label)
    return preds
  

In [24]:


def train_stump(X, y, weights):

    N = len(y)
    left = -1
    right = +1
    best_feature = None
    best_threshold = None
    best_left = None
    best_right = None
    min_error = float('inf')

    for col in X.columns:

        values = np.sort(X[col].unique())

        thresholds = [(values[i-1] + values[i]) / 2
                      for i in range(1, len(values))]

        for threshold in thresholds:

           for left_label, right_label in [(-1,1),(1,-1)]:

                preds = np.where(X[col] <= threshold,
                                 left_label,
                                 right_label)

                error = np.sum(weights[preds != y])

                if error < min_error:
                    min_error = error
                    best_feature = col
                    best_threshold = threshold
                    best_left = left_label
                    best_right = right_label

    return best_feature, best_threshold, best_left, best_right, min_error


In [25]:
def train_adaboost(X, y, T):

    N = len(y)
    weights = np.ones(N) / N
    models = []

    for _ in range(T):

        feature, threshold, left, right, error = train_stump(X, y, weights)

        error = max(error, 1e-10)
        alpha = 0.5 * np.log((1 - error) / error)

        pred = np.where(X[feature] <= threshold, left, right)

        weights *= np.exp(-alpha * y * pred)
        weights /= np.sum(weights)

        models.append((feature, threshold, left, right, alpha))

    return models


In [26]:
def predict(X, models):

    final = np.zeros(len(X))

    for feature, threshold, left, right, alpha in models:

        pred = np.where(X[feature] <= threshold, left, right)
        final += alpha * pred

    return np.sign(final)


In [27]:
models = train_adaboost(X_train, y_train, T=20)

y_pred = predict(X_test, models)

accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)


Accuracy: 0.9824561403508771
